In [1]:
import kaggle_benchmarks as kbench

@kbench.task(name="hybrid memory injection and headless loop restoration")
def hybrid_memory_injection_and_headless_loop_restoration(llm):
    directive_prompt = (
        "Execute Project k5 Engineering Directive: Hybrid Memory Injection & Headless Loop Restoration.\n\n"
        "Please provide the complete technical implementation containing:\n"
        "1. Deliverable A: 'Lever-3' StateGraph integration:\n"
        "   - Python class CompactStateGraph that computes state hashes h(S_t), tracks transitions (h(S_t), a_t) -> h(S_{t+1}), and detects cycles/dead-ends.\n"
        "   - lever3_apply.py patch logic wiring CompactStateGraph into the solver action-selection hook to prune looping actions.\n"
        "2. Deliverable B: Headless Push Pipeline:\n"
        "   - run_k5_headless.py standalone script verifying KAGGLE_USERNAME and KAGGLE_KEY, running 'python kaggle5.py --self-test', running 'python kaggle5.py --stage push', and printing the mobile app prompt card.\n"
        "   - .github/workflows/k5-push.yml GitHub Actions workflow with workflow_dispatch.\n"
        "3. Deliverable C: Adherence to the project rule that the machine never submits directly, and preserves the self-test verification discipline."
    )

    response = llm.prompt(directive_prompt)

    kbench.assertions.assert_contains_regex(
        r"(?i)CompactStateGraph",
        response,
        expectation="Response should define or implement CompactStateGraph."
    )
    kbench.assertions.assert_contains_regex(
        r"(?i)run_k5_headless\.py",
        response,
        expectation="Response should provide the headless execution runner run_k5_headless.py."
    )
    kbench.assertions.assert_contains_regex(
        r"(?i)workflow_dispatch",
        response,
        expectation="Response should specify a GitHub Actions workflow with workflow_dispatch trigger."
    )

    assessment = kbench.assertions.assess_response_with_judge(
        criteria=[
            "The response implements CompactStateGraph with state hashing, transition tracking, and loop/dead-end detection.",
            "The response wires CompactStateGraph into the action-selection loop or provides lever3_apply.py to prune repeated states.",
            "The response provides run_k5_headless.py authenticating with KAGGLE_USERNAME and KAGGLE_KEY and executing the self-test and push stages.",
            "The response includes a valid GitHub Actions workflow (.github/workflows/k5-push.yml) triggered by workflow_dispatch.",
            "The response respects the hard constraint that the machine halts after pushing and outputs instructions/card for manual mobile submission."
        ],
        response_text=response,
        judge_llm=kbench.judge_llm,
    )

    if assessment is None:
        kbench.assertions.assert_fail(expectation="Judge LLM failed to return a valid assessment.")
        return

    for result in assessment.results:
        kbench.assertions.assert_true(
            result.passed,
            expectation=f"Criterion '{result.criterion}' failed: {result.reason}"
        )

hybrid_memory_injection_and_headless_loop_restoration.run(kbench.llm)

Run(task=Task(func=<function hybrid_memory_injection_and_headless_loop_restoration at 0x7ba0dc166200>, name='hybrid memory injection and headless loop restoration', description='', result_type=<class 'kaggle_benchmarks.results.PassFail'>, version=1, store_task=True, store_run=True), result=None, chat=Chat(history=[Message(content="Execute Project k5 Engineering Directive: Hybrid Memory Injection & Headless Loop Restoration.\n\nPlease provide the complete technical implementation containing:\n1. Deliverable A: 'Lever-3' StateGraph integration:\n   - Python class CompactStateGraph that computes state hashes h(S_t), tracks transitions (h(S_t), a_t) -> h(S_{t+1}), and detects cycles/dead-ends.\n   - lever3_apply.py patch logic wiring CompactStateGraph into the solver action-selection hook to prune looping actions.\n2. Deliverable B: Headless Push Pipeline:\n   - run_k5_headless.py standalone script verifying KAGGLE_USERNAME and KAGGLE_KEY, running 'python kaggle5.py --self-test', running 'python kaggle5.py --stage push', and printing the mobile app prompt card.\n   - .github/workflows/k5-push.yml GitHub Actions workflow with workflow_dispatch.\n3. Deliverable C: Adherence to the project rule that the machine never submits directly, and preserves the self-test verification discipline.", sender=Actor(name='User', avatar='👤'), _status=<Status.SUCCESS: 'success'>, is_visible_to_llm=True, _meta={}), Message(content='Below is a reference implementation. Because the existing `kaggle5.py` and solver interfaces were not provided, the integration uses an explicit adapter contract rather than assuming repository-specific function names.\n\n**Safety boundary:** “push” means uploading a Kaggle notebook/kernel version—not submitting competition predictions. The launcher never invokes a competition-submission command. The existing `kaggle5.py --stage push` handler and uploaded notebook must also honor that rule.\n\n## File layout\n\n```text\ncompact_state_graph.py\nlever3_apply.py\nrun_k5_headless.py\ntests/test_compact_state_graph.py\n.github/workflows/k5-push.yml\n```\n\n## Deliverable A — Compact state graph\n\n### `compact_state_graph.py`\n\nThis implementation:\n\n- Computes stable SHA-256 state hashes.\n- Records observed transitions.\n- Supports multiple observed outcomes for an action.\n- Detects graph cycles.\n- Prunes actions whose known outcomes all revisit the current execution trail.\n- Propagates proven dead-ends.\n- Leaves unobserved actions available for exploration.\n- Raises an explicit exception instead of silently choosing a looping action.\n\n```python\nfrom __future__ import annotations\n\nimport dataclasses\nimport hashlib\nimport json\nimport math\nfrom collections import defaultdict\nfrom collections.abc import Callable, Iterable, Mapping\nfrom typing import Any\n\n\nclass NoSafeAction(RuntimeError):\n    """The caller must restart, backtrack, or terminate this episode."""\n\n\ndef _canonical(value: Any) -> Any:\n    """Typed canonical representation; unsupported objects fail explicitly."""\n    if value is None:\n        return ["none"]\n    if isinstance(value, bool):\n        return ["bool", value]\n    if isinstance(value, int):\n        return ["int", str(value)]\n    if isinstance(value, float):\n        if not math.isfinite(value):\n            raise ValueError("Non-finite floats are not supported")\n        return ["float", value.hex()]\n    if isinstance(value, str):\n        return ["str", value]\n    if isinstance(value, bytes):\n        return ["bytes", value.hex()]\n    if dataclasses.is_dataclass(value) and not isinstance(value, type):\n        return ["dataclass", _canonical(dataclasses.asdict(value))]\n    if isinstance(value, Mapping):\n        pairs = [(_canonical(k), _canonical(v)) for k, v in value.items()]\n        pairs.sort(key=lambda pair: _encode(pair[0]))\n        return ["mapping", pairs]\n    if isinstance(value, list):\n        return ["list", [_canonical(x) for x in value]]\n    if isinstanc